# BingePlay
## Streaming Analytics Project

**Week 4 · Advanced SQL · Minor Project**  
**Student:** Smitali Das



In [4]:
%pip install -q pandas sqlalchemy pymysql

import os
import pandas as pd
from sqlalchemy import create_engine

# MySQL connection details
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', 'YOUR_MYSQL_PASSWORD')
MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = os.getenv('MYSQL_PORT', '3306')
MYSQL_DATABASE = 'bingeplay'

engine = create_engine(
    f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
)

print('Database connected')


Database connected


### Q1 — Active revenue


In [ ]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    COALESCE(SUM(monthly_price_inr), 0) AS total_monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""
pd.read_sql(query, engine)


### Q2 — Signup momentum


In [ ]:
query = """
WITH monthly_signups AS (
    SELECT
        MONTH(signup_date) AS month_number,
        DATE_FORMAT(signup_date, '%M') AS month,
        COUNT(*) AS signup_count
    FROM users
    WHERE signup_date >= '2024-01-01'
      AND signup_date < '2024-07-01'
    GROUP BY MONTH(signup_date), DATE_FORMAT(signup_date, '%M')
), ranked AS (
    SELECT
        month_number,
        month,
        signup_count,
        DENSE_RANK() OVER (ORDER BY signup_count DESC) AS signup_rank
    FROM monthly_signups
)
SELECT
    month_number,
    month,
    signup_count,
    CASE WHEN signup_rank = 1 THEN 'Highest signups' ELSE '' END AS note
FROM ranked
ORDER BY month_number;
"""
pd.read_sql(query, engine)


### Q3 — Device analytics


In [ ]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
    ROUND(100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS completion_rate
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""
pd.read_sql(query, engine)


### Q4 — Rating distribution


In [ ]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS rating_percentage,
    ROUND(
        100.0 * SUM(CASE WHEN stars IN (4, 5) THEN COUNT(*) ELSE 0 END) OVER ()
        / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_4_or_5_stars
FROM ratings
GROUP BY stars
ORDER BY stars;
"""
pd.read_sql(query, engine)


### Q5 — Originals vs acquired


In [ ]:
query = """
WITH content_summary AS (
    SELECT
        CASE WHEN is_original = 1 THEN 'Originals' ELSE 'Acquired' END AS content_group,
        COUNT(*) AS number_of_shows,
        ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
        ROUND(AVG(release_year), 2) AS avg_release_year
    FROM shows
    GROUP BY is_original
)
SELECT
    content_group,
    number_of_shows,
    avg_imdb_rating,
    avg_release_year,
    ROUND(
        avg_imdb_rating - MAX(CASE WHEN content_group = 'Acquired' THEN avg_imdb_rating END) OVER (),
        2
    ) AS rating_difference_vs_acquired
FROM content_summary
ORDER BY content_group;
"""
pd.read_sql(query, engine)


### Q6 — Binge day detection


In [ ]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date,
        COUNT(*) AS session_count
    FROM watch_sessions
    WHERE session_date >= '2024-04-01'
      AND session_date < '2024-07-01'
      AND user_id IS NOT NULL
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
), user_binge_counts AS (
    SELECT
        user_id,
        COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
), top_user AS (
    SELECT user_id, binge_days
    FROM user_binge_counts
    ORDER BY binge_days DESC, user_id
    LIMIT 1
)
SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    user_id AS top_user_id,
    binge_days AS top_user_binge_days
FROM top_user;
"""
pd.read_sql(query, engine)


### Q7 — Q1 signups who never watched


In [ ]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(CASE WHEN ws.user_id IS NULL THEN 1 ELSE 0 END) AS never_watched
FROM users u
LEFT JOIN (
    SELECT DISTINCT user_id
    FROM watch_sessions
    WHERE user_id IS NOT NULL
) ws
    ON u.user_id = ws.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-04-01';
"""
pd.read_sql(query, engine)


### Q8 — The over-paying Premium/Family users


In [ ]:
query = """
WITH current_subscription AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT COUNT(*) AS overpaying_users
FROM current_subscription cs
WHERE cs.rn = 1
  AND cs.plan IN ('Premium', 'Family')
  AND NOT EXISTS (
      SELECT 1
      FROM watch_sessions ws
      JOIN shows s ON s.show_id = ws.show_id
      WHERE ws.user_id = cs.user_id
        AND s.min_plan IN ('Premium', 'Family')
  );
"""
pd.read_sql(query, engine)


### Q9 — Upgrade success cohort


In [ ]:
query = """
WITH ordered_subscriptions AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
),
first_subscription AS (
    SELECT *
    FROM ordered_subscriptions
    WHERE rn = 1
),
first_upgrade AS (
    SELECT
        o.user_id,
        MIN(o.start_date) AS first_upgrade_date
    FROM ordered_subscriptions o
    JOIN first_subscription f
        ON f.user_id = o.user_id
    WHERE f.plan = 'Basic'
      AND o.rn > 1
      AND CASE o.plan
            WHEN 'Basic' THEN 1
            WHEN 'Premium' THEN 2
            WHEN 'Family' THEN 3
          END > CASE f.plan
            WHEN 'Basic' THEN 1
            WHEN 'Premium' THEN 2
            WHEN 'Family' THEN 3
          END
    GROUP BY o.user_id
),
still_active AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND (end_date IS NULL OR end_date > '2024-06-30')
),
cohort AS (
    SELECT
        u.user_id,
        u.signup_date,
        fu.first_upgrade_date
    FROM users u
    JOIN first_subscription f ON f.user_id = u.user_id
    JOIN first_upgrade fu ON fu.user_id = u.user_id
    JOIN still_active sa ON sa.user_id = u.user_id
    WHERE u.signup_date >= '2024-01-01'
      AND u.signup_date < '2024-02-01'
      AND f.plan = 'Basic'
)
SELECT
    COUNT(*) AS number_of_users,
    ROUND(AVG(DATEDIFF(first_upgrade_date, signup_date)), 2) AS avg_days_to_first_upgrade
FROM cohort;
"""
pd.read_sql(query, engine)


### Q10 — Cliffhanger comebacks


In [ ]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        a.user_id,
        a.show_id,
        a.session_date AS incomplete_date
    FROM watch_sessions a
    JOIN watch_sessions b
        ON b.user_id = a.user_id
       AND b.show_id = a.show_id
       AND b.session_date BETWEEN DATE_ADD(a.session_date, INTERVAL 1 DAY)
                              AND DATE_ADD(a.session_date, INTERVAL 7 DAY)
    WHERE a.completed = 0
      AND a.user_id IS NOT NULL
),
show_counts AS (
    SELECT
        show_id,
        COUNT(*) AS comeback_events
    FROM comeback_events
    GROUP BY show_id
),
top_show AS (
    SELECT show_id, comeback_events
    FROM show_counts
    ORDER BY comeback_events DESC, show_id
    LIMIT 1
)
SELECT
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
    ts.show_id,
    s.title,
    ts.comeback_events AS top_show_comeback_events
FROM top_show ts
JOIN shows s ON s.show_id = ts.show_id;
"""
pd.read_sql(query, engine)


### Q11 — Consecutive-week engagement


In [ ]:
query = """
WITH distinct_weeks AS (
    SELECT DISTINCT
        user_id,
        YEARWEEK(session_date, 3) AS iso_week,
        DATE_SUB(session_date, INTERVAL WEEKDAY(session_date) DAY) AS week_start
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        week_start,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY week_start
        ) AS rn
    FROM distinct_weeks
),
islands AS (
    SELECT
        user_id,
        week_start,
        DATE_SUB(week_start, INTERVAL rn WEEK) AS island_key
    FROM numbered_weeks
),
streaks AS (
    SELECT
        user_id,
        island_key,
        COUNT(*) AS streak_weeks
    FROM islands
    GROUP BY user_id, island_key
),
qualified AS (
    SELECT DISTINCT user_id
    FROM streaks
    WHERE streak_weeks >= 4
),
longest AS (
    SELECT user_id, streak_weeks
    FROM streaks
    ORDER BY streak_weeks DESC, user_id
    LIMIT 1
)
SELECT
    (SELECT COUNT(*) FROM qualified) AS users_with_4_plus_week_streak,
    l.streak_weeks AS longest_streak_weeks,
    l.user_id AS one_user_with_longest_streak
FROM longest l;
"""
pd.read_sql(query, engine)


### Q12 — Churn signal detection


In [ ]:
query = """
WITH monthly_watch AS (
    SELECT
        user_id,
        SUM(CASE
            WHEN session_date >= '2024-05-01' AND session_date < '2024-06-01'
            THEN watch_minutes ELSE 0 END) AS may_watch_minutes,
        SUM(CASE
            WHEN session_date >= '2024-06-01' AND session_date < '2024-07-01'
            THEN watch_minutes ELSE 0 END) AS june_watch_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id
),
churn_signals AS (
    SELECT
        mw.user_id,
        u.name,
        mw.may_watch_minutes,
        mw.june_watch_minutes,
        ROUND(
            100.0 * (mw.may_watch_minutes - mw.june_watch_minutes)
            / mw.may_watch_minutes,
            2
        ) AS drop_percentage
    FROM monthly_watch mw
    JOIN users u ON u.user_id = mw.user_id
    WHERE mw.may_watch_minutes > 0
      AND mw.june_watch_minutes <= mw.may_watch_minutes * 0.50
)
SELECT
    user_id,
    name,
    may_watch_minutes,
    june_watch_minutes,
    drop_percentage,
    COUNT(*) OVER () AS total_churn_signal_users
FROM churn_signals
ORDER BY drop_percentage DESC, user_id;
"""
pd.read_sql(query, engine)


## Conclusion

The queries above were used to analyse subscriptions, signups, viewing behaviour, ratings, upgrades, engagement and churn for BingePlay.